In [ ]:
import sys
from PyQt5.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QPushButton, QGridLayout, QLabel, QMessageBox
)
from PyQt5.QtGui import QColor
from PyQt5.QtCore import Qt
import serial


class MicroscopeControlApp(QWidget):
    def __init__(self):
        super().__init__()

        # Initialize the UI first
        self.initUI()

        # Then initialize the serial port object
        self.obj = self.init_serial_port()

    def initUI(self):
        """Initializes the UI with buttons and layouts."""
        self.setWindowTitle("Microscope Control")
        self.resize(400, 400)

        # Layouts
        layout = QVBoxLayout()

        # Device connection label (initialize it here)
        self.device_status_label = QLabel("Device: Not connected")
        layout.addWidget(self.device_status_label)

        # Grid layout for Control buttons
        grid_layout = QGridLayout()

        # Create control buttons 1 to 21
        self.controls = []
        for i in range(21):
            control = QPushButton(f"Control {i+1}")
            control.setCheckable(True)
            control.setStyleSheet("background-color: red")
            control.clicked.connect(lambda state, idx=i: self.control_callback(idx, state))
            self.controls.append(control)
            row = i // 3
            col = i % 3
            grid_layout.addWidget(control, row, col)

        layout.addLayout(grid_layout)

        # StopALL button
        self.stop_all_button = QPushButton("Stop All")
        self.stop_all_button.setStyleSheet("background-color: white")
        self.stop_all_button.clicked.connect(self.stop_all_callback)
        layout.addWidget(self.stop_all_button)

        self.setLayout(layout)

    def init_serial_port(self):
        """Initializes the serial port and returns the object."""
        try:
            # Try connecting to COM4 port (modify as per your port)
            obj = serial.Serial('COM8', 9600, timeout=1)
            obj.write(b'')  # Optional: send initial empty message
            obj.flush()
            obj.write_terminator = b'\r'  # Equivalent of 'CR' in MATLAB
            self.show_device_status(True)
            return obj
        except serial.SerialException as e:
            self.show_device_status(False)
            QMessageBox.critical(self, 'Error', 'Could not connect to the device')
            sys.exit(1)

    def show_device_status(self, connected):
        """Updates the device connection status."""
        if connected:
            self.device_status_label.setText("Device correctly connected")
            self.device_status_label.setStyleSheet("color: green")
        else:
            self.device_status_label.setText("Device NOT correctly connected")
            self.device_status_label.setStyleSheet("color: red")

    def control_callback(self, idx, state):
        """Handles Control button toggle actions."""
        if state:
            # Turn the relay on
            self.controls[idx].setStyleSheet("background-color: green")
            self.send_relay_command(f"relay on {idx}")
        else:
            # Turn the relay off
            self.controls[idx].setStyleSheet("background-color: red")
            self.send_relay_command(f"relay off {idx}")

    def stop_all_callback(self):
        """Handles StopALL button press, stopping all relays."""
        self.stop_all_button.setStyleSheet("background-color: green")

        # Turn off all controls
        for i, control in enumerate(self.controls):
            control.setStyleSheet("background-color: red")
            control.setChecked(False)
            self.send_relay_command(f"relay off {i}")

        # Optionally send a command to close all relays at once
        self.send_relay_command('close all')

        # Restore StopALL button to its original color
        self.stop_all_button.setStyleSheet("background-color: white")

    def send_relay_command(self, command):
        """Sends a command to the serial device."""
        if self.obj and self.obj.is_open:
            try:
                self.obj.write(f"{command}\r".encode('utf-8'))
                self.obj.flush()
            except serial.SerialException as e:
                QMessageBox.critical(self, 'Error', 'Failed to communicate with the device')

    def closeEvent(self, event):
        """Overrides close event to ensure the serial connection is closed."""
        if self.obj and self.obj.is_open:
            self.obj.close()
        event.accept()


if __name__ == '__main__':
    app = QApplication(sys.argv)
    ex = MicroscopeControlApp()
    ex.show()
    sys.exit(app.exec_())


In [1]:
import sys
from PyQt5.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QPushButton, QGridLayout, QLabel, QMessageBox
)
from PyQt5.QtGui import QColor
from PyQt5.QtCore import Qt
import serial


class MicroscopeControlApp(QWidget):
    def __init__(self):
        super().__init__()

        # Initialize the UI first
        self.initUI()

        # Then initialize the serial port object
        self.obj = self.init_serial_port()

    def initUI(self):
        """Initializes the UI with buttons and layouts."""
        self.setWindowTitle("Microscope Control")
        self.resize(400, 400)

        # Layouts
        layout = QVBoxLayout()

        # Device connection label (initialize it here)
        self.device_status_label = QLabel("Device: Not connected")
        layout.addWidget(self.device_status_label)

        # Grid layout for Control buttons
        grid_layout = QGridLayout()

        # Create control buttons 1 to 21
        self.controls = []
        for i in range(21):
            control = QPushButton(f"Control {i+1}")
            control.setCheckable(True)
            control.setStyleSheet("background-color: red")
            control.clicked.connect(lambda state, idx=i: self.control_callback(idx, state))
            self.controls.append(control)
            row = i // 3
            col = i % 3
            grid_layout.addWidget(control, row, col)

        layout.addLayout(grid_layout)

        # "Stop All" button
        self.stop_all_button = QPushButton("Stop All")
        self.stop_all_button.setStyleSheet("background-color: black")
        self.stop_all_button.clicked.connect(self.stop_all_callback)
        layout.addWidget(self.stop_all_button)

        # "All On" button
        self.all_on_button = QPushButton("All On")
        self.all_on_button.setStyleSheet("background-color: blue")
        self.all_on_button.clicked.connect(self.all_on_callback)  # New button
        layout.addWidget(self.all_on_button)

        self.setLayout(layout)

    def init_serial_port(self):
        """Initializes the serial port and returns the object."""
        try:
            # Try connecting to COM4 port (modify as per your port)
            obj = serial.Serial('COM8', 9600, timeout=1)
            obj.write(b'')  # Optional: send initial empty message
            obj.flush()
            obj.write_terminator = b'\r'  # Equivalent of 'CR' in MATLAB
            self.show_device_status(True)
            return obj
        except serial.SerialException as e:
            self.show_device_status(False)
            QMessageBox.critical(self, 'Error', 'Could not connect to the device')
            sys.exit(1)

    def show_device_status(self, connected):
        """Updates the device connection status."""
        if connected:
            self.device_status_label.setText("Device correctly connected")
            self.device_status_label.setStyleSheet("color: green")
        else:
            self.device_status_label.setText("Device NOT correctly connected")
            self.device_status_label.setStyleSheet("color: red")

    def control_callback(self, idx, state):
        """Handles Control button toggle actions."""
        if state:
            # Turn the relay on
            self.controls[idx].setStyleSheet("background-color: green")
            self.send_relay_command(f"relay on {idx}")
        else:
            # Turn the relay off
            self.controls[idx].setStyleSheet("background-color: red")
            self.send_relay_command(f"relay off {idx}")

    def stop_all_callback(self):
        """Handles StopALL button press, stopping all relays."""
        self.stop_all_button.setStyleSheet("background-color: green")

        # Turn off all controls
        for i, control in enumerate(self.controls):
            control.setStyleSheet("background-color: red")
            control.setChecked(False)
            self.send_relay_command(f"relay off {i}")

        # Optionally send a command to close all relays at once
        self.send_relay_command('close all')

        # Restore StopALL button to its original color
        self.stop_all_button.setStyleSheet("background-color: black")

    def all_on_callback(self):
        """Handles the 'All On' button press, turning on all relays."""
        self.all_on_button.setStyleSheet("background-color: green")

        # Turn on all controls
        for i, control in enumerate(self.controls):
            control.setStyleSheet("background-color: green")
            control.setChecked(True)
            self.send_relay_command(f"relay on {i}")

        # Optionally send a command to open all relays at once
        self.send_relay_command('open all')

    def send_relay_command(self, command):
        """Sends a command to the serial device."""
        if self.obj and self.obj.is_open:
            try:
                self.obj.write(f"{command}\r".encode('utf-8'))
                self.obj.flush()
            except serial.SerialException as e:
                QMessageBox.critical(self, 'Error', 'Failed to communicate with the device')

    def closeEvent(self, event):
        """Overrides close event to ensure the serial connection is closed."""
        if self.obj and self.obj.is_open:
            self.obj.close()
        event.accept()


if __name__ == '__main__':
    app = QApplication(sys.argv)
    ex = MicroscopeControlApp()
    ex.show()
    sys.exit(app.exec_())


SystemExit: 0

c:\ProgramData\anaconda3\envs\bootcamp_20240819\Lib\site-packages\IPython\core\interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
